# Peek at read-based taxonomy ↔ biosamples

Worked-example queries linking read-based taxonomy results
(Kraken2, GOTTCHA2, Centrifuge) to biosamples via
`nmdc_metadata.biosample_to_workflow_run`.

**Both directions:**
- §1–3: biosample → what taxa were detected (by method)
- §4–5: taxon name → which biosamples detected it (by method)

**Centrifuge** data is not yet ingested. §3 documents the expected
schema and query pattern for when it is.

In [1]:
spark = get_spark_session(app_name="peek_read_taxonomy_links", tenant_name="nmdc")
print(f"Spark version: {spark.version}")

Spark version: 4.0.1


### Preflight: check available read taxonomy tables

In [2]:
existing = {r["tableName"] for r in spark.sql("SHOW TABLES IN nmdc_results").toPandas().to_dict("records")}

checks = {
    'kraken2':   'kraken2_classification_report',
    'gottcha2':  'gottcha2_classification_report',
    'centrifuge': 'centrifuge_output_report_file',
}

available = {}
for key, tbl in checks.items():
    found = tbl in existing
    available[key] = found
    status = "OK" if found else "MISSING"
    print(f"{status}: nmdc_results.{tbl}")

OK: nmdc_results.kraken2_classification_report
OK: nmdc_results.gottcha2_classification_report
OK: nmdc_results.centrifuge_output_report_file


### Pick an anchor biosample

Find a biosample that has at least one Kraken2 result via the workflow bridge.

In [3]:
_anchor_df = spark.sql("""
    SELECT b2wr.biosample_id, b2wr.workflow_run_id, b2wr.workflow_type
    FROM   nmdc_results.kraken2_classification_report k
    JOIN   nmdc_metadata.biosample_to_workflow_run b2wr
             ON b2wr.workflow_run_id = k.workflow_run_id
    LIMIT 1
""").toPandas()
if _anchor_df.empty:
    raise RuntimeError("no kraken2_classification_report row joins to biosample_to_workflow_run — verify both tables are populated")
anchor = _anchor_df.iloc[0]
BIOSAMPLE_ID  = anchor["biosample_id"]
KRAKEN2_RUN_ID = anchor["workflow_run_id"]
print(anchor.to_string())

biosample_id                 nmdc:bsm-12-crjz4m94
workflow_run_id          nmdc:wfrbt-11-aak5c206.1
workflow_type      nmdc:ReadBasedTaxonomyAnalysis


### 1. Biosample → Kraken2 taxa

Species-level hits (rank = 'S') with at least 1 direct read, ordered by % abundance.
Kraken2 has 36M rows total — always filter by workflow_run_id.

In [4]:
spark.sql(f"""
    SELECT rank, name, taxid, pct_clade, clade_reads, direct_reads
    FROM   nmdc_results.kraken2_classification_report
    WHERE  workflow_run_id = '{KRAKEN2_RUN_ID}'
      AND  rank = 'S'
      AND  direct_reads > 0
    ORDER BY pct_clade DESC
    LIMIT 20
""").toPandas()

,rank,name,taxid,pct_clade,clade_reads,direct_reads
0,S,Sphingobium hydrophobicum,1673076,0.24,53920,53920
1,S,...,9606,0.17,37055,37055
2,S,Sphingopyxis fribergensis,1515612,0.06,12196,12196
3,S,Flavobacterium psychrophilum,96345,0.05,10767,9583
4,S,Variovorax paradoxus,34073,0.05,10611,2034
5,S,Rhodoferax saidenbachensis,1484693,0.04,8808,8808
6,S,Polaromonas sp. SP1,2268087,0.04,9766,9766
7,S,Polaromonas sp. JS666,296591,0.04,8347,8347
8,S,Limnohabitans sp. 63ED37-2,1678128,0.04,8740,8740
9,S,Ramlibacter tataouinensis,94132,0.04,8865,4336


### 2. Biosample → GOTTCHA2 taxa

Species-level hits ordered by relative abundance. GOTTCHA2 uses a unique linear-length
scoring approach that reduces false positives vs. Kraken2.

In [5]:
if not available['gottcha2']:
    print("SKIP: gottcha2_classification_report not available")
else:
    g2_run = spark.sql(f"""
        SELECT b2wr.workflow_run_id
        FROM   nmdc_metadata.biosample_to_workflow_run b2wr
        JOIN   nmdc_results.gottcha2_classification_report g
                 ON g.workflow_run_id = b2wr.workflow_run_id
        WHERE  b2wr.biosample_id = '{BIOSAMPLE_ID}'
        LIMIT 1
    """).toPandas()
    if g2_run.empty:
        print(f"No GOTTCHA2 results for {BIOSAMPLE_ID} — try a different biosample")
        GOTTCHA2_RUN_ID = None
    else:
        GOTTCHA2_RUN_ID = g2_run.iloc[0]["workflow_run_id"]
        display(spark.sql(f"""
            SELECT LEVEL, NAME, TAXID, READ_COUNT, REL_ABUNDANCE
            FROM   nmdc_results.gottcha2_classification_report
            WHERE  workflow_run_id = '{GOTTCHA2_RUN_ID}'
              AND  LEVEL = 'species'
            ORDER BY REL_ABUNDANCE DESC
            LIMIT 20
        """).toPandas())

,LEVEL,NAME,TAXID,READ_COUNT,REL_ABUNDANCE
0,species,Sphingobium xenophagum,121428,61814,0.338692
1,species,Limnobacter sp. SAORIC-580,2732163,62442,0.329841
2,species,Chamaesiphon minutus,1173032,32424,0.059426
3,species,Candidatus Karelsulcia muelleri,336810,451,0.055425
4,species,Pseudomonas aeruginosa,287,417,0.035340
5,species,Sphingobium yanoikuyae,13690,1985,0.027642
6,species,Sheathia arcuata (plastid),340433_pt,160,0.019913
7,species,Pseudomonas alcaliphila,101564,626,0.017801
8,species,Acidovorax temperans,80878,1471,0.009690
9,species,Pseudomonas chengduensis,489632,203,0.008850


### 3. Biosample → Centrifuge taxa

`nmdc_results.centrifuge_output_report_file` — 20,564,570 rows, one per (taxon, workflow run).

| Column | Type | Description |
|---|---|---|
| `name` | string | Taxon name |
| `taxID` | int | NCBI taxon ID |
| `taxRank` | string | Taxonomic rank |
| `genomeSize` | int | Reference genome size |
| `numReads` | int | Reads classified to this taxon |
| `numUniqueReads` | int | Reads uniquely classified here |
| `abundance` | float | Fraction of total reads |
| `workflow_run_id` | string | Join key to `biosample_to_workflow_run` |

In [6]:
if not available['centrifuge']:
    print("SKIP: centrifuge_output_report_file not available")
else:
    centrifuge_run = spark.sql(f"""
        SELECT b2wr.workflow_run_id
        FROM   nmdc_metadata.biosample_to_workflow_run b2wr
        JOIN   nmdc_results.centrifuge_output_report_file c
                 ON c.workflow_run_id = b2wr.workflow_run_id
        WHERE  b2wr.biosample_id = '{BIOSAMPLE_ID}'
        LIMIT 1
    """).toPandas()
    if centrifuge_run.empty:
        print(f"No Centrifuge results for {BIOSAMPLE_ID}")
    else:
        run_id = centrifuge_run.iloc[0]["workflow_run_id"]
        display(spark.sql(f"""
            SELECT name, taxID, taxRank, numReads, numUniqueReads, abundance
            FROM   nmdc_results.centrifuge_output_report_file
            WHERE  workflow_run_id = '{run_id}'
              AND  abundance > 0
            ORDER BY abundance DESC
            LIMIT 20
        """).toPandas())

,name,taxID,taxRank,numReads,numUniqueReads,abundance
0,Sphingobium hydrophobicum,1673076,species,57081,55109,0.633040
1,Pseudomonas alcaliphila,101564,species,34109,27531,0.116210
2,Pseudomonas pseudoalcaligenes,330,species,21130,14193,0.089327
3,Sphingopyxis terrae,33052,species,7477,4433,0.027586
4,Sphingopyxis alaskensis,117207,species,7742,5272,0.010763
5,Pseudomonas sp. LPH1,1898684,species,13374,9511,0.008554
6,Sphingomonas koreensis,93064,species,4677,2687,0.007432
7,Candidatus Sulcia muelleri,336810,species,469,420,0.007273
8,Pseudomonas stutzeri,316,species,11046,6468,0.005624
9,Sphingopyxis sp. QXT-31,1357916,species,8493,5325,0.005214


### 4. Taxon → biosamples (Kraken2)

Which biosamples had a given taxon detected? Filter by name and a minimum clade percentage.
Aggregates across all Kraken2 runs for each biosample.

In [7]:
TARGET_TAXON = "Bacteroides"  # substitute any taxon name

if not available['kraken2']:
    print("SKIP: kraken2_classification_report not available")
else:
    display(spark.sql(f"""
        SELECT b2wr.biosample_id,
               bsm.env_broad_scale_term_id,
               bsm.geo_loc_name_has_raw_value,
               MAX(k.pct_clade)   AS max_pct_clade,
               SUM(k.clade_reads) AS total_clade_reads
        FROM   nmdc_results.kraken2_classification_report k
        JOIN   nmdc_metadata.biosample_to_workflow_run b2wr
                 ON b2wr.workflow_run_id = k.workflow_run_id
        JOIN   nmdc_metadata.biosample_set bsm
                 ON bsm.id = b2wr.biosample_id
        WHERE  TRIM(k.name) = '{TARGET_TAXON}'
          AND  k.pct_clade > 0.1
        GROUP BY b2wr.biosample_id, bsm.env_broad_scale_term_id, bsm.geo_loc_name_has_raw_value
        ORDER BY max_pct_clade DESC
        LIMIT 20
    """).toPandas())

,biosample_id,env_broad_scale_term_id,geo_loc_name_has_raw_value,max_pct_clade,total_clade_reads
0,nmdc:bsm-11-06ms9b03,ENVO:01000253,"Germany: Brandenburg, Berlin-Heidemühle",0.15,285852


### 5. Taxon → biosamples (GOTTCHA2)

In [8]:
if not available['gottcha2']:
    print("SKIP: gottcha2_classification_report not available")
else:
    display(spark.sql(f"""
        SELECT b2wr.biosample_id,
               bsm.env_broad_scale_term_id,
               bsm.geo_loc_name_has_raw_value,
               MAX(g.REL_ABUNDANCE)  AS max_rel_abundance,
               SUM(g.READ_COUNT)     AS total_read_count
        FROM   nmdc_results.gottcha2_classification_report g
        JOIN   nmdc_metadata.biosample_to_workflow_run b2wr
                 ON b2wr.workflow_run_id = g.workflow_run_id
        JOIN   nmdc_metadata.biosample_set bsm
                 ON bsm.id = b2wr.biosample_id
        WHERE  TRIM(g.NAME) = '{TARGET_TAXON}'
          AND  g.REL_ABUNDANCE > 0
        GROUP BY b2wr.biosample_id, bsm.env_broad_scale_term_id, bsm.geo_loc_name_has_raw_value
        ORDER BY max_rel_abundance DESC
        LIMIT 20
    """).toPandas())

,biosample_id,env_broad_scale_term_id,geo_loc_name_has_raw_value,max_rel_abundance,total_read_count
0,nmdc:bsm-11-06ms9b03,ENVO:01000253,"Germany: Brandenburg, Berlin-Heidemühle",0.029220,140015
1,nmdc:bsm-11-m6sxmb47,ENVO:01000253,"Italy: Piedmont, San Benigno Canavese",0.017459,66406
2,nmdc:bsm-11-eqp67n25,ENVO:01000253,"USA: Connecticut, Simsbury",0.011019,76875
3,nmdc:bsm-11-kmew8154,ENVO:01000253,USA: Nebraska,0.001811,9180
4,nmdc:bsm-12-0tm4k954,ENVO:03605008,"USA: Kansas, Kings Creek",0.001527,23
5,nmdc:bsm-11-1e5j5h70,ENVO:01000253,"USA: Oregon, Blue River",0.001516,141
6,nmdc:bsm-11-gzbkd269,ENVO:01000253,"USA: Massachusetts, Burlington",0.001133,1755
7,nmdc:bsm-11-f1w93z37,ENVO:01000253,USA: Virginia,0.000924,6352
8,nmdc:bsm-11-5g2kjf44,ENVO:00000446,USA: Oregon,0.000655,1929
9,nmdc:bsm-11-7eemvx60,ENVO:01000253,USA: Iowa,0.000519,2105


### 6. Taxon → biosamples (Centrifuge)

Which biosamples had a given taxon detected by Centrifuge?
Aggregates across all Centrifuge runs for each biosample.

In [9]:
if not available['centrifuge']:
    print("SKIP: centrifuge_output_report_file not available")
else:
    display(spark.sql(f"""
        SELECT b2wr.biosample_id,
               bsm.env_broad_scale_term_id,
               bsm.geo_loc_name_has_raw_value,
               MAX(c.abundance)        AS max_abundance,
               SUM(c.numReads)         AS total_reads
        FROM   nmdc_results.centrifuge_output_report_file c
        JOIN   nmdc_metadata.biosample_to_workflow_run b2wr
                 ON b2wr.workflow_run_id = c.workflow_run_id
        JOIN   nmdc_metadata.biosample_set bsm
                 ON bsm.id = b2wr.biosample_id
        WHERE  TRIM(c.name) = '{TARGET_TAXON}'
          AND  c.numReads > 0
        GROUP BY b2wr.biosample_id, bsm.env_broad_scale_term_id, bsm.geo_loc_name_has_raw_value
        ORDER BY max_abundance DESC
        LIMIT 20
    """).toPandas())

,biosample_id,env_broad_scale_term_id,geo_loc_name_has_raw_value,max_abundance,total_reads
